# 19 · Phase profiling of HybridFIM at k = 3, 4 and 5  (VM only, ~3.5–4.5 h)

**Question.** Why does HybridFIM still win some higher-order workloads and lose others? Scaling
(notebook 15) shows *how* runtime changes; this notebook measures *where* the time goes inside one
HybridFIM call:

| phase | what it covers |
|---|---|
| input encoding | Python: item encoder, transaction ids, plus the `find_associations` wrapper |
| index construction | Rust: CSR inverted index (transaction → items, item → transactions) |
| pair counting | Rust: scatter pair counting (the k = 2 kernel) |
| extension setup | Rust: per-item roaring bitmaps and the TID-list of every frequent pair |
| extension k = 3, 4, 5 | Rust, per level: projection (transactions filtered to frequent items) + anchor-and-extend (counting, intersections, merge) |
| output construction | Rust → NumPy arrays, then Python rule expansion and DataFrame |
| other | memory release at the end of the Rust call |

The phases add up to the runtime the harness records (`inner_wall_s`) for the same run.

**Panel (declared before running).** Support 1e-4, k = 3, 4 and 5, one and eight threads. A run at
k includes every level below it, so the k = 5 runs also show how levels 3 and 4 compare inside one call.

| workloads | role in the May 2026 grid |
|---|---|
| T10.D100k, T20.D1M, T25.D5M | the only May k ≥ 3 wins with a non-trivial result (1.1k–1.6k frequent 3-itemsets) |
| T30.D2M, Instacart, Chainstore | contrasting losses (Instacart: a pairwise win that reverses at k ≥ 3) |

36 profiled HybridFIM configurations = 144 executions (1 warm-up + 3 measured), plus **plain
fastapriori 0.2.0** (the venv, no timers) on the same configurations, interleaved run by run (the
overhead check), and Borgelt Eclat on the 18 workload/k combinations as the fixed native reference
(total runtime, with its file-reading share).

**Lattice size is part of the contrast.** Of the 51 k ≥ 3 cells HybridFIM won in May, 45 had no
frequent k-itemsets and 3 had fewer than 1,000; the three winners here are the only ones with more.
Their levels ≥ 4 are (near-)empty (T25.D5M: 0 at k = 4, 5; T20.D1M: 5 at k = 4), while the losses
have tens of thousands. Section 4 prints these counts; section 10 compares *normalised* per-level
costs, which is what can carry an implementation claim.

**Instrumented build.** `lib/fastapriori-0.2.1`, built with its Cargo feature `phaseprof`, is fastapriori 0.2.0 plus coarse phase
timers in the default k ≥ 3 path (`rust_compute_pipeline_v3_adaptive`, about ten clock reads per
call) and an opt-in profile written when `FASTAPRIORI_PROFILE` names a file. Results are identical
to 0.2.0 (checked below). It is installed into `_work/phaseprof_site`, **not** into the venv, so
every other notebook keeps using 0.2.0. Diff: `diff -r lib/fastapriori-0.2.0 lib/fastapriori-0.2.1`. (The October 2026 campaign ran the same
instrumentation compiled in unconditionally, built as version `0.2.0+phaseprof`; see
`executed/19_phase_profiling.ipynb`.)

**Limits.** Only HybridFIM is instrumented: Eclat's runtime is context, and no claim about Eclat's
internal phases can follow from this notebook. Work counters are cheap ones (sizes of maps, summed
TID-list lengths); nothing is counted inside the hot loop.

**Never run this while another benchmark runs on the same VM** (the guard below refuses).
Outputs: `results/runs/<campaign>/phaseprof.csv` (one row per run, harness format),
`phaseprof_phases.jsonl` (one profile per profiled run) and tables/figures `phaseprof_*` in the
same folder.

`FA_MODE=replay` analyses the saved campaign (`vm_20261029` unless `FA_CAMPAIGN` says otherwise): no mining and no data needed; tables and figures go to `results/derived/<campaign>/`.


In [ ]:
import os, sys, json, math, time, hashlib, shutil, subprocess, tempfile
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start the notebook from <bundle>/notebooks"
REPLAY = os.environ.get("FA_MODE") == "replay"    # analyse the saved campaign: no mining, nothing overwritten
assert REPLAY or sys.platform.startswith("linux"), "VM / WSL only: every run is timed with /usr/bin/time under ulimit"
assert "fastapriori" not in sys.modules and "exp_runners" not in sys.modules, \
    "restart the kernel and run from the top: the profiling build must be first on the path"
os.environ.setdefault("FA_CAMPAIGN", "vm_20261029" if REPLAY else "new_campaign")

# The instrumented library goes into its own folder; putting it first on sys.path (this kernel) and on
# PYTHONPATH (the fresh process of every run) selects it, and FA_LIB_DIR makes the harness hash its
# sources into the build id. The production venv is not touched.
PROF_SRC = BUNDLE / "lib" / "fastapriori-0.2.1"
WORK = Path(os.environ.get("FA_WORKDIR", BUNDLE / "_work"))
PROF_SITE = WORK / "phaseprof_site"
sys.path[:0] = [str(PROF_SITE), str(BUNDLE / "bench")]
os.environ["PYTHONPATH"] = os.pathsep.join([str(PROF_SITE)] + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p])
os.environ["FA_LIB_DIR"] = str(PROF_SRC)
os.environ.pop("FASTAPRIORI_PROFILE", None)

import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("campaign", os.environ["FA_CAMPAIGN"], "| profiling build source", PROF_SRC.relative_to(BUNDLE))
if os.cpu_count() != 8 and not REPLAY:
    print(f"NOTE: this machine has {os.cpu_count()} CPUs; the panel declares 8 threads (c3-highmem-8). "
          "Multi-threaded runs use all CPUs and are labelled with the count actually used.")

## 1 · Guard: nothing else may run

The build compiles Rust on all cores and the runs are timed, so both refuse to start while another
benchmark (harness driver, Borgelt/LCM binary, Rust build) is running or the machine is loaded.
Set `ALLOW_BUSY = True` only if you know why it is busy.

In [ ]:
ALLOW_BUSY = False

def busy_report():
    me = str(os.getpid())
    ps = subprocess.run(["ps", "-eo", "pid=,args="], capture_output=True, text=True).stdout
    pats = ("exp_driver.py", "/tools/bin/", "/tools/bin_2026-05/", "lcm ", "cargo build", "rustc ")
    procs = [l.strip() for l in ps.splitlines() if any(p in l for p in pats) and l.split()[0] != me]
    load1 = os.getloadavg()[0]
    return procs, load1

def guard(max_wait_s=300):
    """Refuse if a benchmark process runs; if only the load is high (e.g. right after the build),
    wait up to max_wait_s for it to settle."""
    t0 = time.time()
    while True:
        procs, load1 = busy_report()
        if ALLOW_BUSY or (not procs and load1 <= 1.5):
            print(f"idle enough: 1-min load {load1:.2f}, no benchmark processes")
            return
        if procs or time.time() - t0 > max_wait_s:
            raise RuntimeError(f"machine busy (1-min load {load1:.2f}); running:\n  " + "\n  ".join(procs[:10] or ["-"]))
        time.sleep(15)

if not REPLAY:
    guard()

## 2 · Build the instrumented library (once; ~3–5 min)

Rebuilt only when the source changed (a hash of `lib/fastapriori-0.2.1` is stored next to
the build). Same toolchain as `env/setup_vm.sh` (Rust 1.95.0).

In [ ]:
def tree_hash(root):
    h = hashlib.sha256()
    for p in sorted(root.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts:
            h.update(p.relative_to(root).as_posix().encode()); h.update(p.read_bytes())
    return h.hexdigest()[:16]

src_hash = tree_hash(PROF_SRC)
stamp = PROF_SITE / ".source_sha"
if REPLAY:
    print("replay: no build")
elif stamp.exists() and stamp.read_text() == src_hash:
    print("instrumented build up to date:", PROF_SITE)
else:
    guard()
    target = Path(tempfile.mkdtemp(prefix="cargo_phaseprof_", dir=WORK))
    env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
    env.update(RUSTUP_TOOLCHAIN="1.95.0", CARGO_TARGET_DIR=str(target), MATURIN_PEP517_ARGS="--features phaseprof",
               PATH=f"{Path.home() / '.cargo' / 'bin'}{os.pathsep}{os.environ['PATH']}")
    shutil.rmtree(PROF_SITE, ignore_errors=True)
    t0 = time.time()
    b = subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--target", str(PROF_SITE), str(PROF_SRC)],
                       env=env, capture_output=True, text=True)
    shutil.rmtree(target, ignore_errors=True)
    print(b.stdout[-800:], b.stderr[-2000:])
    assert b.returncode == 0, "build failed"
    stamp.write_text(src_hash)
    print(f"built in {time.time() - t0:.0f}s ->", PROF_SITE)

In [ ]:
import importlib
# PROF_SITE did not exist when it was put on sys.path, and Python caches "no importer" for a missing
# folder; drop that entry so the fresh build is found.
sys.path_importer_cache.pop(str(PROF_SITE), None)
importlib.invalidate_caches()
import exp_runners as pr
import exp_data as pdd
OUT = BUNDLE / "results" / "derived" / pr.CAMPAIGN if REPLAY else pr.RESULTS
OUT.mkdir(parents=True, exist_ok=True)
print("campaign", pr.CAMPAIGN, "->", pr.RESULTS)
if not REPLAY:
    import fastapriori, importlib.metadata as md

    assert Path(fastapriori.__file__).resolve().is_relative_to(PROF_SITE.resolve()), fastapriori.__file__
    assert Path(pr.ext_path()).resolve().is_relative_to(PROF_SITE.resolve()), pr.ext_path()
    # the fresh process of every run must see the same build
    child = subprocess.run([sys.executable, "-c", "import fastapriori, importlib.metadata as m; print(m.version('fastapriori'), fastapriori.__file__)"],
                           capture_output=True, text=True, env=os.environ).stdout.strip()
    assert child.startswith("0.2.1 "), child
    print("kernel :", md.version("fastapriori"), fastapriori.__file__)
    print("runs   :", child)
    print("build  ", pr.build_id())

    # The production reference: fastapriori as installed in the venv (what every other notebook runs).
    # A child without the profiling build on its path reports version, origin and build id.
    PROD_ENV_DROP = ("PYTHONPATH", "FA_LIB_DIR", "FASTAPRIORI_PROFILE")
    PROBE = "\n".join([
        "import sys, json, hashlib, importlib.metadata as m",
        "sys.path.insert(0, sys.argv[1])",
        "import exp_runners as pr, fastapriori",
        "d = m.distribution('fastapriori'); u = d.read_text('direct_url.json')",
        "print(json.dumps(dict(version=d.version, file=fastapriori.__file__, direct_url=json.loads(u) if u else None,",
        "    ext=pr.ext_path(), ext_sha256=hashlib.sha256(open(pr.ext_path(), 'rb').read()).hexdigest(),",
        "    build_id=pr.build_id())))",
    ])
    prod_probe = subprocess.run([sys.executable, "-c", PROBE, str(BUNDLE / "bench")], capture_output=True, text=True,
                                env={k: v for k, v in os.environ.items() if k not in PROD_ENV_DROP})
    assert prod_probe.returncode == 0, prod_probe.stderr[-1500:]
    PROD_INFO = json.loads(prod_probe.stdout.strip().splitlines()[-1])
    assert PROD_INFO["version"] == "0.2.0" and PROD_INFO["build_id"] != pr.build_id(), PROD_INFO
    print("production reference:", json.dumps(PROD_INFO, indent=1))

## 3 · Same results as fastapriori 0.2.0

T10.D100k, s = 1e-4, k = 3, 4 and 5: the full rule tables from the production build (the venv) and from
the instrumented build with profiling off and on must be identical. Each runs in its own process.

In [ ]:
CHECK = r"""
import json, os, sys
import pandas as pd
from fastapriori import find_associations
src, out, k = sys.argv[1], sys.argv[2], int(sys.argv[3])
df = pd.read_parquet(src)
r = find_associations(df, transaction_col=df.columns[0], item_col=df.columns[1], k=k, min_support=1e-4,
                      min_confidence=0.0, backend="auto", algo="fast", sorted_by=None, low_memory=False)
cols = [f"antecedent_{i}" for i in range(1, k)] + ["consequent"]
r.sort_values(cols).reset_index(drop=True).to_parquet(out)
"""
if not REPLAY:
    chk = Path(tempfile.mkdtemp(prefix="phaseprof_check_", dir=WORK))
    src = pr.parquet_for("T10.I4.D100k.N25k.L25k")
    prod_env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "FA_LIB_DIR", "FASTAPRIORI_PROFILE")}
    for k in (3, 4, 5):
        frames = {}
        for label, env in (("0.2.0", prod_env), ("phaseprof off", dict(os.environ)),
                           ("phaseprof on", dict(os.environ, FASTAPRIORI_PROFILE=str(chk / f"p{k}.json")))):
            out = chk / f"{label.replace(' ', '_')}_{k}.parquet"
            r = subprocess.run([sys.executable, "-c", CHECK, str(src), str(out), str(k)], env=env, capture_output=True, text=True)
            assert r.returncode == 0, r.stderr[-1500:]
            frames[label] = pd.read_parquet(out)
        same = {lbl: frames["0.2.0"].equals(f) for lbl, f in frames.items() if lbl != "0.2.0"}
        print(f"k={k}: {len(frames['0.2.0'])} rules; identical to 0.2.0: {same}")
        assert all(same.values())
    print("profile written when enabled:", json.loads((chk / "p5.json").read_text())["levels"])
    shutil.rmtree(chk, ignore_errors=True)

## 4 · Panel and its roles in the May 2026 grid

The roles are declared; the table recomputes them from the saved May medians (HybridFIM, eight
threads, against the fastest external miner) so the reader can check them.

In [ ]:
PANEL = {   # paper name: (registry name, declared role)
    "T10.D100k":  ("T10.I4.D100k.N25k.L25k", "win"),
    "T20.D1M":    ("T20.I4.D1000k.N50k.L50k", "win"),
    "T25.D5M":    ("T25.I8.D5000k.N40k.L40k", "win"),
    "T30.D2M":    ("T30.I8.D2000k.N10k.L10k", "loss"),
    "Instacart":  ("Instacart", "loss"),
    "Chainstore": ("Chainstore", "loss"),
}
S, KS = 1e-4, [3, 4, 5]
NAME = {reg: paper for paper, (reg, _) in PANEL.items()}
ROLE = {paper: role for paper, (_, role) in PANEL.items()}

may = pd.read_csv(BUNDLE / "results" / "campaign_2026-05" / "results_rigorous_20260523_115442.csv")
may = may[may.dataset.isin(NAME) & may.k.isin([2] + KS) & np.isclose(may.min_support, S)]
piv = may.pivot_table(index=["dataset", "k"], columns="method", values="wall_s_median")
ext = [c for c in piv.columns if c.startswith(("borgelt_", "pyfim_"))]
nk = may[may.method == "borgelt_eclat"].set_index(["dataset", "k"]).n_itemsets_median
ctx = pd.DataFrame({"hybridfim_8t_s": piv["fastapriori_fast"], "best_external_s": piv[ext].min(axis=1),
                    "best_external": piv[ext].idxmin(axis=1), "borgelt_eclat_s": piv["borgelt_eclat"],
                    "n_frequent_k": nk})
ctx["ratio_vs_best"] = ctx.hybridfim_8t_s / ctx.best_external_s
ctx["observed"] = np.where(ctx.ratio_vs_best < 1, "win", "loss")
ctx = ctx.reset_index(); ctx["workload"] = ctx.dataset.map(NAME); ctx["declared"] = ctx.workload.map(ROLE)
display(ctx[["workload", "k", "n_frequent_k", "hybridfim_8t_s", "best_external", "best_external_s", "ratio_vs_best", "observed", "declared"]].round(3))
mism = ctx[(ctx.k >= 3) & (ctx.observed != ctx.declared)]
print("declared roles that the May k>=3 medians do not support:", "none" if mism.empty else mism[["workload", "k"]].values.tolist())

## 5 · Runs (resumable)

Order: per workload and k, the five methods back to back, so the instrumented and the production
build run next to each other in time and drift does not bias the overhead check. Each run is a
fresh process under the harness (`pr.run_once`: `ulimit -v`, `timeout`, `/usr/bin/time -v`). A
profiled run receives `FASTAPRIORI_PROFILE=<file>`; its profile is appended to
`phaseprof_phases.jsonl` (before the CSV row, so a resumed run never lacks its profile). A
production run is started without the profiling build on `PYTHONPATH` and recorded with the
production build id. Rerunning the cell skips finished runs.

| recorded method | harness method | threads | library | profile |
|---|---|---|---|---|
| `hyb_prof` | `hyb` | all (8 on the VM) | instrumented | on |
| `hyb_prof_1t` | `hyb_1t` | 1 | instrumented | on |
| `hyb` | `hyb` | all (8 on the VM) | 0.2.0 (venv) | none (overhead reference) |
| `hyb_1t` | `hyb_1t` | 1 | 0.2.0 (venv) | none (overhead reference) |
| `borgelt_eclat` | `borgelt_eclat` | 1 (native) | Borgelt binary | n/a |

In [ ]:
EXPERIMENT = "phaseprof"
METHODS = {"hyb_prof": "hyb", "hyb_prof_1t": "hyb_1t", "hyb": "hyb", "hyb_1t": "hyb_1t", "borgelt_eclat": "borgelt_eclat"}
PROFILED = {"hyb_prof", "hyb_prof_1t"}
PRODUCTION = {"hyb", "hyb_1t"}      # run with the venv's fastapriori 0.2.0
REPEATS, WARMUP, TIMEOUT_S = 3, 1, 1500
PHASES_JSONL = pr.RESULTS / f"{EXPERIMENT}_phases.jsonl"
DATASETS = [reg for reg, _ in PANEL.values()]
print(f"{len(DATASETS) * len(KS) * len(METHODS)} cells x {WARMUP + REPEATS} runs")

In [ ]:
from datetime import datetime, timezone
from tqdm.auto import tqdm

if REPLAY:
    bid = json.loads(PHASES_JSONL.read_text(encoding="utf-8").splitlines()[-1])["build_id"]
else:
    guard()
    pr.preflight(sorted(set(METHODS.values())))
    bid, host = pr.build_id(), os.uname().nodename
    done = pr.load_results(EXPERIMENT)
    other = sorted(set(done.get("build_id", pd.Series(dtype=str)).dropna().astype(str)) - {bid, PROD_INFO["build_id"]})
    assert not other, f"{pr.results_path(EXPERIMENT)} holds rows of build(s) {other}; start a new FA_CAMPAIGN"
    pr.write_manifest(EXPERIMENT, dict(datasets=DATASETS, methods=METHODS, ks=KS, supports=[S], repeats=REPEATS,
                                       warmup=WARMUP, timeout_s=TIMEOUT_S, profiled=sorted(PROFILED),
                                       production=sorted(PRODUCTION), production_build=PROD_INFO,
                                       lib=str(PROF_SRC.relative_to(BUNDLE)), lib_source_sha=src_hash), bid)

    cells = [(ds, k, m) for ds in DATASETS for k in KS for m in METHODS]
    prof_file = pr.WORK / f"phaseprof_{os.getpid()}.json"
    bar = tqdm(total=len(cells), desc=EXPERIMENT, unit="cell", dynamic_ncols=True)
    for ds, k, rec in cells:
        bar.set_postfix_str(f"{NAME[ds]} | {rec} | k={k}", refresh=True)
        have, failed = pr._cell_state(done, ds, rec, k, S)
        if failed:
            bar.update(1); continue
        for run_id in range(WARMUP + REPEATS):
            if run_id in have:
                continue
            prof_file.unlink(missing_ok=True)
            pythonpath = os.environ.get("PYTHONPATH")
            if rec in PROFILED:
                os.environ["FASTAPRIORI_PROFILE"] = str(prof_file)
            if rec in PRODUCTION:
                os.environ.pop("PYTHONPATH", None)          # the child imports the venv's fastapriori
            try:
                r = pr.run_once(METHODS[rec], ds, k, S, timeout_s=TIMEOUT_S)
            finally:
                os.environ.pop("FASTAPRIORI_PROFILE", None)
                if pythonpath is not None:
                    os.environ["PYTHONPATH"] = pythonpath
            base = dict(experiment=EXPERIMENT, dataset=ds, method=rec, k=k, min_support=S, run_id=run_id,
                        warmup=run_id < WARMUP, build_id=PROD_INFO["build_id"] if rec in PRODUCTION else bid,
                        host=host, timestamp_utc=datetime.now(timezone.utc).isoformat())
            r.update(base)
            if rec in PROFILED and r["status"] == "ok":
                prof = json.loads(prof_file.read_text()) if prof_file.exists() else None
                if prof is None:
                    bar.write(f"WARNING: no profile written for {NAME[ds]} {rec} k={k} run {run_id}")
                with PHASES_JSONL.open("a", encoding="utf-8") as f:    # before the CSV row (resume safety)
                    f.write(json.dumps(dict(base, status=r["status"], inner_wall_s=r["inner_wall_s"],
                                            process_wall_s=r["process_wall_s"], n_itemsets=r["n_itemsets"],
                                            profile=prof), default=float) + "\n")
            pr._append(EXPERIMENT, r)
            bar.write(f"[{EXPERIMENT}] {NAME[ds]} | {rec} | k={k} run {run_id}: {r['status']} "
                      f"wall={r['wall_s']} rss={r['peak_rss_kb']}")
            if r["status"] != "ok":
                break
        bar.update(1)
    bar.close()
    prof_file.unlink(missing_ok=True)

## 6 · Phases per run

Each profiled run becomes one row of phase times that add up to its `inner_wall_s`:

* `input_encoding` = Python encoding + the `find_associations` wrapper around it
* `index`, `pairs`, `ext_setup` (pair TID-lists), `ext_k3`, `ext_k4`, `ext_k5` (projection + anchor-extend)
* `output` = Rust → NumPy + Python decode
* `other` = the rest of the Rust call (memory release)

`input_encoding` and `other` are remainders, so the phases add up to `inner_wall_s` by construction
(the printed difference is a bookkeeping identity, not a check). A workload/k/threads cell is
represented by its **median measured run** (the run whose total is the median of the three), so the
stacked phases are one real run. Only profiles of the current instrumented build are used, one per
run (the last, if a run was repeated), and cells without exactly `REPEATS` measured profiles are listed.

In [ ]:
raw = [json.loads(l) for l in PHASES_JSONL.read_text(encoding="utf-8").splitlines() if l.strip()]
stale = sum(r["build_id"] != bid for r in raw)
latest = {(r["dataset"], r["method"], r["k"], r["run_id"]): r for r in raw if r["build_id"] == bid}
print(f"{len(raw)} profile lines; {stale} from other builds ignored; {len(raw) - stale - len(latest)} duplicates dropped")
rows = []
for rec in latest.values():
    p = rec.get("profile")
    if not p or rec["status"] != "ok":
        continue
    lv = {l["k"]: l for l in p["levels"]}
    row = dict(workload=NAME[rec["dataset"]], dataset=rec["dataset"], k=rec["k"], threads=p["threads"],
               method=rec["method"], run_id=rec["run_id"], warmup=rec["warmup"], inner_wall_s=rec["inner_wall_s"],
               n_itemsets=rec["n_itemsets"], n_rules=p["n_rules"], n_freq_pairs=p["n_freq_pairs"],
               n_transactions=p["n_transactions"], n_items=p["n_items"])
    levels_s = sum(l["project_s"] + l["extend_s"] for l in p["levels"])
    row.update(input_encoding=p["encode_s"] + (rec["inner_wall_s"] - p["total_s"]),
               index=p["index_s"], pairs=p["pairs_s"], ext_setup=p["pair_tids_s"],
               output=p["output_rust_s"] + p["decode_s"],
               other=p["rust_call_s"] - (p["index_s"] + p["pairs_s"] + p["pair_tids_s"] + levels_s + p["output_rust_s"]))
    for kk in KS:
        l = lv.get(kk)
        row[f"ext_k{kk}"] = (l["project_s"] + l["extend_s"]) if l else 0.0
        for c in ("project_s", "extend_s", "n_anchors", "anchor_tids", "n_projected", "n_frequent"):
            row[f"k{kk}_{c}"] = l[c] if l else np.nan
    rows.append(row)
runs = pd.DataFrame(rows)
EXT_COLS = [f"ext_k{kk}" for kk in KS]
PHASE_COLS = ["input_encoding", "index", "pairs", "ext_setup"] + EXT_COLS + ["output", "other"]
runs["phase_sum"] = runs[PHASE_COLS].sum(axis=1)
runs["unaccounted"] = runs.inner_wall_s - runs.phase_sum
print(f"{len(runs)} profiled runs; bookkeeping identity |inner_wall - sum of phases| <= {runs.unaccounted.abs().max():.1e} s")

meas = runs[~runs.warmup.astype(bool)]
n_meas = meas.groupby(["workload", "k", "threads"]).size()
print("cells without exactly REPEATS measured profiles:", n_meas[n_meas != REPEATS].to_dict() or "none")
def median_run(g):
    g = g.sort_values("inner_wall_s")
    return g.iloc[(len(g) - 1) // 2]
cell = meas.groupby(["workload", "k", "threads"], sort=False).apply(median_run, include_groups=False).reset_index()
cell["role"] = cell.workload.map(ROLE)
cell.to_csv(OUT / "phaseprof_cells.csv", index=False)
display(cell[["workload", "role", "k", "threads", "inner_wall_s"] + PHASE_COLS].round(3))

## 7 · Overhead: instrumented vs production 0.2.0

Median total runtime of the instrumented build with profiling on, against plain fastapriori 0.2.0
(the venv, no timers), measured interleaved on the same cells. A cell is flagged only when the
difference of medians exceeds **both** `NOTICEABLE` (relative) and the larger run-to-run range of
the two (noise); flagged cells should not be split into phases without a separate check.

In [ ]:
NOTICEABLE = 0.03
res = pr.load_results(EXPERIMENT)
res = res[(res.status == "ok") & (~res.warmup.astype(bool))]
g = res.groupby(["dataset", "k", "method"]).wall_s
med, rng = g.median().unstack("method"), (g.max() - g.min()).unstack("method")
parts, flags = [], []
for tag, prof, prod in (("mt", "hyb_prof", "hyb"), ("1t", "hyb_prof_1t", "hyb_1t")):
    d = med[prof] - med[prod]
    noise = np.maximum(rng[prof], rng[prod])
    parts.append(pd.DataFrame({f"{tag}_instrumented_s": med[prof], f"{tag}_0.2.0_s": med[prod],
                               f"{tag}_ratio": med[prof] / med[prod], f"{tag}_noise_s": noise}))
    flags.append((d.abs() > noise) & ((d / med[prod]).abs() > NOTICEABLE))
ov = pd.concat(parts, axis=1)
ov["flagged"] = pd.concat(flags, axis=1).any(axis=1)
ov = ov.reset_index(); ov.insert(0, "workload", ov.dataset.map(NAME)); ov = ov.drop(columns="dataset")
ov.to_csv(OUT / "phaseprof_overhead.csv", index=False)
display(ov.round(3))
print(f"ratio range: {ov[['mt_ratio', '1t_ratio']].min().min():.3f} .. {ov[['mt_ratio', '1t_ratio']].max().max():.3f}")
print("flagged cells:", ov.loc[ov.flagged, ["workload", "k"]].values.tolist() or "none")

## 8 · Result agreement

Itemset counts must agree between the instrumented build, fastapriori 0.2.0 and Borgelt Eclat in every cell.

In [ ]:
summ = pr.summarize(pr.load_results(EXPERIMENT))
bad = pr.check_agreement(summ)
print("count disagreements:", len(bad))
if len(bad):
    display(bad)
display(summ.pivot_table(index=["dataset", "k"], columns="method", values="n_itemsets").rename(index=NAME, level=0))

## 9 · Stacked runtime figure

Each bar is HybridFIM's runtime **relative to Borgelt Eclat on the same workload and k** (Eclat's
process wall = 1, dashed line), split into phases. Eclat's time includes reading its text input
(HybridFIM's includes encoding its in-memory input); the short dotted tick on each bar marks Eclat
without reading its input (wall − Borgelt's "reading" step). Borgelt mines *during* its "writing"
step, so the harness column `borgelt_mine_s` (total − reading − writing) is only item recoding and
transaction sorting and is not used here. Top: one thread (like-for-like with
single-threaded Eclat); bottom: all threads (8 on the VM) against the same single-threaded Eclat,
the configuration of the paper's comparisons. Tick labels give the number of frequent k-itemsets:
read every bar together with its lattice size. Seconds are in the table below.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PHASE_LABEL = {"input_encoding": "input encoding", "index": "index construction", "pairs": "pair counting",
               "ext_setup": "extension setup (pair TID-lists)", "output": "output construction",
               "other": "other (memory release)", **{f"ext_k{kk}": f"extension k = {kk}" for kk in KS}}
# categorical slots in their fixed order, one per phase in pipeline order (validated for k = 3..5:
# CVD-adjacent separation and normal-vision floor pass); "other" is a neutral grey. Three slots are
# below 3:1 contrast against the surface, hence the table view below.
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
assert len(PHASE_COLS) - 1 <= len(SLOTS), "more phases than validated categorical slots"
PHASE_COLOR = dict(zip(PHASE_COLS[:-1], SLOTS)) | {"other": "#a3a29b"}
INK, MUTED, SURFACE = "#1f1f1d", "#6b6a64", "#fcfcfb"

ecl_runs = res[res.method == "borgelt_eclat"].assign(excl_read_s=lambda d: d.wall_s - d.borgelt_read_s)
ecl = ecl_runs.groupby(["dataset", "k"])
eclat = ecl.wall_s.median()
eclat_read = ecl.borgelt_read_s.median()
eclat_noread = ecl.excl_read_s.median()
nfreq = {(w, k): cell.loc[(cell.workload == w) & (cell.k == k), f"k{k}_n_frequent"].max() for w in PANEL for k in KS}
THREADS = sorted(cell.threads.unique())
order = [w for w in PANEL if ROLE[w] == "win"] + [w for w in PANEL if ROLE[w] == "loss"]
xs, labels, x = [], [], 0.0
for i, w in enumerate(order):
    if i and ROLE[w] != ROLE[order[i - 1]]:
        x += 0.8                                    # gap between winners and losses
    for k in KS:
        nf = nfreq.get((w, k))
        xs.append((w, k, x)); labels.append(f"{w}\nk = {k}\n|F| = {nf:,.0f}" if pd.notna(nf) else f"{w}\nk = {k}"); x += 1.0
    x += 0.35

fig, axes = plt.subplots(len(THREADS), 1, figsize=(4.0 + 0.75 * len(xs), 3.8 * len(THREADS)), sharex=True,
                         facecolor=SURFACE, squeeze=False)
axes = axes[:, 0]
for ax, threads in zip(axes, THREADS):
    ax.set_facecolor(SURFACE)
    for w, k, xpos in xs:
        c = cell[(cell.workload == w) & (cell.k == k) & (cell.threads == threads)]
        e = eclat.get((PANEL[w][0], k), np.nan)
        if c.empty or not np.isfinite(e):
            ax.text(xpos, 0.05, "n/a", ha="center", color=MUTED, fontsize=8); continue
        c = c.iloc[0]; bottom = 0.0
        for ph in PHASE_COLS:
            h = c[ph] / e
            if h > 0:
                ax.bar(xpos, h, bottom=bottom, width=0.78, color=PHASE_COLOR[ph], edgecolor=SURFACE, linewidth=1.0)
                bottom += h
        m = eclat_noread.get((PANEL[w][0], k), np.nan) / e
        if np.isfinite(m):
            ax.plot([xpos - 0.39, xpos + 0.39], [m, m], color=INK, linestyle=(0, (1, 1.5)), linewidth=1.2)
        ax.annotate(f"{c.inner_wall_s:.1f}s\n{bottom:.2f}x", (xpos, bottom), xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=7, color=INK)
    ax.axhline(1.0, color=INK, linestyle=(0, (4, 3)), linewidth=1.0)
    ax.set_ylabel(f"HybridFIM {threads} thread{'s' if threads > 1 else ''}\nvs single-threaded Eclat\n(runtime / Eclat wall)", color=INK)
    ax.tick_params(colors=MUTED); ax.grid(axis="y", color="#e4e3dd", linewidth=0.6); ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color("#c9c8c1")
axes[-1].set_xticks([p for *_, p in xs]); axes[-1].set_xticklabels(labels, fontsize=8, color=INK)
wins_end = max(p for w, _, p in xs if ROLE[w] == "win")
for ax in axes:
    ax.axvline(wins_end + 0.95, color="#c9c8c1", linewidth=0.8)
    ax.margins(y=0.12)
for role, title in (("win", "May 2026 k ≥ 3 wins"), ("loss", "contrasting losses")):
    pos = [p for w, _, p in xs if ROLE[w] == role]
    if pos:
        axes[0].text(np.mean(pos), 1.04, title, transform=axes[0].get_xaxis_transform(), ha="center",
                     va="bottom", color=MUTED, fontsize=9)
fig.legend(handles=[Patch(color=PHASE_COLOR[p], label=PHASE_LABEL[p]) for p in PHASE_COLS], loc="upper center",
           ncol=4, frameon=False, fontsize=8, bbox_to_anchor=(0.5, 1.0), labelcolor=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
for ext in ("pdf", "png"):
    fig.savefig(OUT / f"phaseprof_stacked.{ext}", dpi=200, facecolor=SURFACE)
plt.show()

In [ ]:
# table view of the figure: seconds per phase, total, Eclat, ratio
tab = cell.copy()
key = [(PANEL[w][0], k) for w, k in zip(tab.workload, tab.k)]
tab["eclat_s"] = [eclat.get(x, np.nan) for x in key]
tab["eclat_read_s"] = [eclat_read.get(x, np.nan) for x in key]
tab["eclat_excl_read_s"] = [eclat_noread.get(x, np.nan) for x in key]
tab["ratio_vs_eclat"] = tab.inner_wall_s / tab.eclat_s
tab["ratio_vs_eclat_excl_read"] = tab.inner_wall_s / tab.eclat_excl_read_s
tab["n_frequent_k"] = [nfreq.get((w, k)) for w, k in zip(tab.workload, tab.k)]
tab = tab.set_index(["role", "workload", "k", "threads"])[["n_frequent_k"] + PHASE_COLS + [
    "inner_wall_s", "eclat_s", "eclat_read_s", "eclat_excl_read_s", "ratio_vs_eclat", "ratio_vs_eclat_excl_read"]]
tab.to_csv(OUT / "phaseprof_table.csv")
display(tab.round(3))

## 10 · Shares, lattice size and normalised costs

Phase shares per cell, shown next to the lattice size (`n_frequent` per level), then the work
counters per level: `n_anchors` (frequent (k−1)-itemsets extended), `anchor_tids` (their summed
TID-list length, i.e. transactions scanned), `n_projected` (transactions left after projection),
`n_frequent` (frequent k-itemsets found). The normalised costs (extension ns per scanned
transaction, projection ns per input transaction) are what can be compared across workloads; raw
shares mostly reflect how much there is to mine. There is deliberately no average over "winners"
and "losses": the winners' higher levels are (near-)empty, so such an average describes the data,
not the implementation.

In [ ]:
share = cell.set_index(["role", "workload", "k", "threads"])[PHASE_COLS].div(cell.set_index(["role", "workload", "k", "threads"]).inner_wall_s, axis=0)
share["dominant"] = share[PHASE_COLS].idxmax(axis=1).map(PHASE_LABEL)
share["pairs+setup"] = share.pairs + share.ext_setup
share["extension"] = share[EXT_COLS].sum(axis=1)
lat = cell.set_index(["role", "workload", "k", "threads"])[[f"k{kk}_n_frequent" for kk in KS]]
display(lat.join((share.drop(columns="dominant") * 100).round(1)).join(share.dominant))

work = cell[["role", "workload", "k", "threads", "n_freq_pairs", "n_rules"] +
            [c for c in cell.columns if c.startswith(tuple(f"k{kk}_" for kk in KS))]].copy()
ntx = cell["n_transactions"]
for kk in KS:
    work[f"k{kk}_extend_ns_per_tid"] = 1e9 * work[f"k{kk}_extend_s"] / work[f"k{kk}_anchor_tids"].replace(0, np.nan)
    work[f"k{kk}_project_ns_per_txn"] = 1e9 * work[f"k{kk}_project_s"] / ntx
work.to_csv(OUT / "phaseprof_work.csv", index=False)
display(work.round(4))

**Writing it up.** What the data can carry:
* *Where HybridFIM's time goes* in each cell (shares), always quoted with the lattice size: a phase
  that dominates because the higher levels are empty is a property of the workload.
* *Implementation claims across workloads* only from the normalised costs (ns per scanned TID, ns
  per transaction), e.g. whether extension is more expensive per unit of work on the losses.
* *Against Eclat*: total runtime only, with and without Eclat's file reading (`ratio_vs_eclat`,
  `ratio_vs_eclat_excl_read`); nothing about Eclat's internal phases.
A statement such as "on the winning workloads pair construction dominates and extension adds little;
on the losing workloads extension dominates" must hold in every cell it covers (both thread counts)
and must say that the winners' levels ≥ 4 are (near-)empty. Report exceptions as they are.